# Crafting an AI-Powered HR Assistant: Nestlé HR Policy Chatbot

## Objective
This project builds a conversational HR Assistant for Nestlé using LangChain, OpenAI GPT, and Gradio.
The chatbot extracts and processes HR policy documents, enabling natural-language queries and accurate, document-based answers.

---
## 1. Import Essential Libraries and Setup

In [ ]:
# !pip install --upgrade langchain openai faiss-cpu gradio PyPDF2

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

# LangChain and related
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Gradio UI
import gradio as gr

---
## 2. Configure OpenAI API Environment

Before running, ensure your OpenAI API key is configured.
You can export it as an environment variable or set it directly inside the notebook.

In [ ]:
# export OPENAI_API_KEY or set in notebook:
# os.environ["OPENAI_API_KEY"] = ""
# if "OPENAI_API_KEY" not in os.environ:
#     raise EnvironmentError("OPENAI_API_KEY must be set in environment before running this notebook")

<small>Disabled openAI API config code because LMS has it inbuilt</small>

---
## 3. Load and Process Nestlé's HR Policy Document

Load the HR policy PDF using LangChain’s PyPDFLoader to extract text content for downstream processing.

> **Note:**  
> This notebook assumes the HR policy PDF file is stored in the same directory as the notebook.  
> The expected path is:  `hr-policy-en.pdf`
> Update `pdf_path` in the code if file is stored elsewhere.

In [ ]:
pdf_path = "hr-policy-en.pdf"  # path to HR policy document

try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    if len(documents) == 0:
        raise ValueError("PDF loaded but no pages returned")
    total_chars = sum(len(d.page_content) for d in documents)
    print(f"Loaded {len(documents)} pages; total characters: {total_chars:,}")
except Exception as e:
    raise RuntimeError(f"Failed to load PDF at {pdf_path}: {e}")

---
## 4. Split Documents into Manageable Chunks

Long documents must be split into smaller segments (chunks) so embeddings and retrieval work effectively.
Each chunk overlaps slightly to preserve sentence continuity.

In [ ]:
# Configure text splitter for optimal chunk size
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

In [ ]:
# Split documents into chunks
text_chunks = text_splitter.split_documents(documents)
if len(text_chunks) == 0:
    raise RuntimeError("Text splitter returned zero chunks")
avg_chunk_size = sum(len(c.page_content) for c in text_chunks) // len(text_chunks)
print(f"Created {len(text_chunks)} chunks, average size {avg_chunk_size} chars")

---
## 5. Create Text Embeddings and FAISS Vector Store

Convert each text chunk into numerical vectors using OpenAI embeddings.
Store these embeddings in FAISS, a fast similarity-search database.

In [ ]:
# Create FAISS index
embeddings = OpenAIEmbeddings()
print("Creating FAISS index (this will call OpenAI embeddings for chunks)...")
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [ ]:
# persist to disk
index_dir = "faiss_index"
os.makedirs(index_dir, exist_ok=True)
vectorstore.save_local(index_dir)
print(f"FAISS index saved to {index_dir}")

---
## 6. Initialize the GPT Model

Instantiate the ChatOpenAI model to handle natural-language queries and produce concise answers.

In [ ]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.2, max_tokens=600)

---
## 7. Define Prompt Template

The prompt template guides GPT to use only the HR document context, ensuring accurate and compliant answers.

In [ ]:
prompt_template = """
You are Nestlé's HR Assistant, a helpful and professional AI trained to answer employee questions using only the official Nestlé HR policy documents.

Your role is to:
- Provide clear, concise, and accurate answers based solely on the provided HR policy context.
- Avoid speculation or assumptions. If the answer is not explicitly found in the context, say: "This information is not available in the HR policy documents."
- Maintain a formal, respectful, and supportive tone appropriate for HR communication.
- When possible, cite page numbers from the source documents to support your answers.

Instructions:
- Do not use external knowledge or personal opinions.
- Do not fabricate answers or cite policies not present in the context.
- Do not repeat the question in your response.
- If the question is ambiguous or unclear, ask for clarification.
- If multiple relevant sections exist, summarize them concisely and cite all applicable pages.

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

---
## 8. Build the Retrieval-Based QA System

Combine GPT with the FAISS retriever using LangChain’s RetrievalQA chain.
This system first retrieves the most relevant chunks and then formulates an answer.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True,
)

---
## 9. Define Chatbot Query Function

This function handles user input, retrieves answers, and includes source page references.

In [ ]:
def answer_query(user_message, chat_history):

    if chat_history is None:
        chat_history = []

    try:
        res = qa_chain({"query": user_message})
        answer_text = res.get("result") or res.get("answer") or ""
        source_docs = res.get("source_documents", [])

        pages = []
        for doc in source_docs:
            md = getattr(doc, "metadata", {}) or {}
            p = md.get("page")
            if isinstance(p, int):
                pages.append(p + 1)

        pages = sorted(set(pages))
        if pages:
            page_info = "Pages: " + ", ".join(str(p) for p in pages)
            answer_text = answer_text.rstrip() + f"\n\nSource: {page_info}"

        chat_history.append(("User", user_message))
        chat_history.append(("Assistant", answer_text))

        return chat_history, ""

    except Exception as e:
        err = f"Error while answering: {e}"
        chat_history.append(("User", user_message))
        chat_history.append(("Assistant", err))
        return chat_history, ""

---
## 10. Build Gradio Interface

Create an interactive UI using Gradio Blocks.
The interface provides a chat window for user interaction and displays results with source references.

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Nestlé HR Assistant (RAG)")
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox(placeholder="Ask about Nestlé HR policies...")
    state = gr.State([])
    submit = gr.Button("Send")

    def submit_handler(user_message, history):
        if history is None:
            history = []
        try:
            res = qa_chain({"query": user_message})
            answer_text = res.get("result") or res.get("answer") or ""
            source_docs = res.get("source_documents", [])
            pages = []
            for doc in source_docs:
                md = getattr(doc, "metadata", {}) or {}
                p = md.get("page")
                if isinstance(p, int):
                    pages.append(p + 1)
            if pages:
                page_info = "Pages: " + ", ".join(str(p) for p in sorted(set(pages)))
                answer_text = answer_text.rstrip() + f"\n\nSource: {page_info}"
        except Exception as e:
            answer_text = f"Error while answering: {e}"

        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": answer_text})
        return history, ""

    submit.click(fn=submit_handler, inputs=[msg, state], outputs=[chatbot, msg])
    msg.submit(fn=submit_handler, inputs=[msg, state], outputs=[chatbot, msg])

---
## 11. Launch the Chatbot

In [ ]:
demo.launch(share=True)